## Notebook objective
This notebook builds the lemma sample used for the experimental analysis. It starts from the Sensocomune inventory, applies linguistic and frequency filters, and produces a balanced dataset across semantic-complexity clusters for definition-generation experiments.

## Input
This step loads the reference lemma list from the Sensocomune resource file.

### Study design
The sampling procedure selects a balanced and representative set of lemmas for downstream experiments. Starting from the Sensocomune inventory, it retains nouns and verbs, filters them by lexical prominence, and distributes them across semantic-complexity clusters.

In [ ]:
import pandas as pd

df: pd.DataFrame = pd.read_json(
    # File not shared due to copyright issues
    "../resources/sensocomune/new-tdm-lemmario.json", orient="records"
)
df.head()

,lemma,marca,categorie,numero
0,a,AD,[s.f. e m.inv.],1.0
1,a,FO,[prep.],2.0
2,abbagliante,AD,"[p.pres., agg., s.m.]",NaN
3,abbaiare,AD,[v.intr. e tr.],NaN
4,abbandonare,FO,[v.tr.],NaN


## First filter: nouns and verbs
This step excludes all entries that are not nouns or verbs.

In [4]:
sostantivi_filter: pd.Series = df["categorie"].apply(lambda x: any(c.startswith("s.") for c in x))
verbi_filter: pd.Series = df["categorie"].apply(lambda x: any(c.startswith("v.") for c in x))
filtered_df: pd.DataFrame = df[sostantivi_filter | verbi_filter]
filtered_df.head()

,lemma,marca,categorie,numero
0,a,AD,[s.f. e m.inv.],1.0
2,abbagliante,AD,"[p.pres., agg., s.m.]",NaN
3,abbaiare,AD,[v.intr. e tr.],NaN
4,abbandonare,FO,[v.tr.],NaN
5,abbandonato,AU,"[p.pass., agg., s.m.]",NaN


In [5]:
filtered_df["lemma"].__len__()

6469

## Second filter: lemma mark
This step keeps lemmas with high usage and strong availability.

### Filtering rationale
The second filter prioritizes frequent and well-documented lemmas to ensure that the sample is linguistically relevant and suitable for reliable definition generation.

In [6]:
fondamentali_filter: pd.Series = filtered_df["marca"] == "FO"

filtered_df = filtered_df[fondamentali_filter]
filtered_df.head()

,lemma,marca,categorie,numero
4,abbandonare,FO,[v.tr.],NaN
7,abbassare,FO,[v.tr.],NaN
19,abbracciare,FO,[v.tr.],1.0
28,abitare,FO,"[v.intr. e tr., s.m.]",NaN
30,abito,FO,[s.m.],NaN


In [7]:
len(df) - len(filtered_df), len(filtered_df)

(5485, 1761)

## Definition retrieval
Once the lemmas are selected, the full dictionary entries are retrieved.

In [8]:
import sys

sys.path.append("../../")

In [9]:
import json
from pathlib import Path

data: list[dict] = json.loads(
    Path("../../resources/sensocomune/tdm.base.json").read_text()
)
print(f"Loaded {len(data)} items")

Loaded 6056 items


In [10]:
# Filter out items that are not in the filtered_df
data_set: dict[str, dict] = {item["lemma"]: item for item in data}
intersecated_data: set[str] = set(filtered_df["lemma"]).intersection(data_set.keys())
print(f"Found {len(intersecated_data)} items in the original data")

Found 1538 items in the original data


In [11]:
def fix_lemma_definition(lemma: str) -> dict:
    lemma_dict: dict = data_set.get(lemma)
    lemma_dict["marca"] = "FO"
    lemma_dict["categorie"] = lemma_dict["grammar"].copy()
    return lemma_dict

selected_lemmas = list(
    map(fix_lemma_definition, [lemma for lemma in intersecated_data if lemma in data_set])
)
len(selected_lemmas)

1538

In [12]:
from IPython.display import display, Markdown
import json

display(
    Markdown(
        f"```json\n{json.dumps(selected_lemmas[-1], indent=2)}\n```"
    )
)

```json
{
  "_id": {
    "$oid": "648aaea469ad0a50d065579a"
  },
  "source": "paraviaXML/27104.xml",
  "lemma": "controllo",
  "grammar": [
    "s.m."
  ],
  "discr": 0,
  "acceptations": [
    {
      "id": "1",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Il controllare e il suo risultato",
              "source": "tdm"
            }
          ],
          "examples": [
            "il controllo dei passaporti",
            "dei bagagli",
            "evitare il controllo doganale"
          ]
        },
        {
          "number": 2,
          "marks": [
            "estensivo"
          ],
          "definitions": [
            {
              "glossa": "Luogo in cui avviene l\u2019ispezione o sono fatti gli accertamenti",
              "source": "tdm"
            }
          ],
          "examples": [
            "fermarsi",
            "attendere al controllo"
          ]
        },
        {
          "number": 3,
          "marks": [],
          "definitions": [
            {
              "glossa": "Una visita medica",
              "source": "tdm"
            }
          ],
          "examples": [
            "fare un controllo medico"
          ]
        }
      ]
    },
    {
      "id": "2a",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [
            "eccetera"
          ],
          "definitions": [
            {
              "glossa": "Verifica del funzionamento e dell\u2019efficienza di materiali, macchinari, impianti, eccetera",
              "source": "tdm"
            }
          ],
          "examples": [
            "fare il controllo di una caldaia",
            "sottoporre un impianto a controlli periodici"
          ]
        }
      ]
    },
    {
      "id": "2b",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [
            "specialmente"
          ],
          "definitions": [
            {
              "glossa": "Vigilanza, sorveglianza, specialmente militare o poliziesca",
              "source": "tdm"
            }
          ],
          "examples": [
            "zona sotto il controllo dell\u2019ONU"
          ]
        },
        {
          "number": 2,
          "marks": [],
          "definitions": [
            {
              "glossa": "Chi \u00e8 incaricato di controllare",
              "source": "tdm"
            }
          ],
          "examples": [
            "a che ora passa il c.?"
          ]
        }
      ]
    },
    {
      "id": "2c",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Funziona da aggettivogeno",
              "source": "tdm"
            }
          ],
          "examples": [
            "controllo radar"
          ]
        }
      ]
    },
    {
      "id": "3a",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Dominio, padronanza",
              "source": "tdm"
            }
          ],
          "examples": [
            "mantiene sempre il controllo delle proprie emozioni",
            "mantenere il controllo della situazione"
          ]
        }
      ]
    },
    {
      "id": "3b",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Dominio di s\u00e9, autocontrollo",
              "source": "tdm"
            }
          ],
          "examples": [
            "non perde mai il controllo"
          ]
        }
      ]
    },
    {
      "id": "3c",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [
            "eccetera"
          ],
          "definitions": [
            {
              "glossa": "Capacit\u00e0 di manovrare un autoveicolo, una nave, un aereo, eccetera , azionandone i comandi",
              "source": "tdm"
            }
          ],
          "examples": [
            "mantenere",
            "perdere",
            "riprendere il controllo del mezzo"
          ]
        }
      ]
    },
    {
      "id": "4",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Potere di chi, detenendo la maggioranza delle azioni di una societ\u00e0, pu\u00f2 deciderne la politica e le attivit\u00e0",
              "source": "tdm"
            }
          ],
          "examples": []
        }
      ]
    },
    {
      "id": "5",
      "usage": "AU",
      "field": "",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Attivit\u00e0 di un organismo statale o di un ufficio di un\u2019azienda privata al fine di disciplinare un determinato settore",
              "source": "tdm"
            }
          ],
          "examples": [
            "il controllo dei prezzi",
            "il controllo sulle esportazioni",
            "il controllo degli investimenti"
          ]
        }
      ]
    },
    {
      "id": "6",
      "usage": "TS",
      "field": "tecnica, tecnologia",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Tecnica, tecnologia , dispositivo per regolare il funzionamento di un apparecchio",
              "source": "tdm"
            }
          ],
          "examples": [
            "il controllo del tono",
            "dei bassi"
          ]
        }
      ]
    },
    {
      "id": "7",
      "usage": "TS",
      "field": "simile",
      "grammar": [],
      "is_link": false,
      "senses": [
        {
          "number": 1,
          "marks": [],
          "definitions": [
            {
              "glossa": "Elettronica , dispositivo, programma, eccetera , in grado di segnalare guasti, errori o disfunzioni di un\u2019apparecchiatura, un sistema e simile",
              "source": "tdm"
            }
          ],
          "examples": []
        }
      ]
    }
  ],
  "locutions": [],
  "marca": "FO",
  "categorie": [
    "s.m."
  ]
}
```

In [13]:
selected_lemmas_df: pd.DataFrame = pd.DataFrame(
    selected_lemmas
)
selected_lemmas_df.head()

,_id,source,lemma,grammar,discr,acceptations,locutions,marca,categorie
0,{'$oid': '648aaf3869ad0a50d065d1ed'},paraviaXML/98317.xml,ritardo,[s.m.],0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,[s.m.]
1,{'$oid': '648aaec769ad0a50d06574bf'},paraviaXML/43984.xml,fine,[s.f. e m.],0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,[s.f. e m.]
2,{'$oid': '648aaebf69ad0a50d0656e1a'},paraviaXML/40359.xml,esercito,[s.m.],0,"[{'id': '1a', 'usage': 'FO', 'field': '', 'gra...",[],FO,[s.m.]
3,{'$oid': '648aae0669ad0a50d064b6f9'},paraviaXML/106412.xml,sereno,"[agg., s.m.]",0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,"[agg., s.m.]"
4,{'$oid': '648aafa169ad0a50d066264b'},paraviaXML/4648.xml,alzare,[v.tr.],0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,[v.tr.]


In [14]:
selected_lemmas_df["acceptations_count"] = selected_lemmas_df["acceptations"].apply(lambda x: len(x))
selected_lemmas_df["acceptations_count"].describe()

count    1538.000000
mean        9.712614
std         6.801057
min         1.000000
25%         5.000000
50%         8.000000
75%        13.000000
max        71.000000
Name: acceptations_count, dtype: float64

In [15]:
from typing import Literal


def extract_cluster(acc_count: int):
    cluster: Literal["FIRST", "SECOND", "THIRD", "FOURTH"]
    if 1 <= acc_count < 5:
        cluster = "FIRST"
    elif 5 <= acc_count < 8:
        cluster = "SECOND"
    elif 8 <= acc_count < 13:
        cluster = "THIRD"
    elif 13 <= acc_count:
        cluster = "FOURTH"
    return cluster

selected_lemmas_df["cluster"] = selected_lemmas_df["acceptations_count"].apply(extract_cluster)
selected_lemmas_df.head()

,_id,source,lemma,grammar,discr,acceptations,locutions,marca,categorie,acceptations_count,cluster
0,{'$oid': '648aaf3869ad0a50d065d1ed'},paraviaXML/98317.xml,ritardo,[s.m.],0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,[s.m.],13,FOURTH
1,{'$oid': '648aaec769ad0a50d06574bf'},paraviaXML/43984.xml,fine,[s.f. e m.],0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,[s.f. e m.],14,FOURTH
2,{'$oid': '648aaebf69ad0a50d0656e1a'},paraviaXML/40359.xml,esercito,[s.m.],0,"[{'id': '1a', 'usage': 'FO', 'field': '', 'gra...",[],FO,[s.m.],4,FIRST
3,{'$oid': '648aae0669ad0a50d064b6f9'},paraviaXML/106412.xml,sereno,"[agg., s.m.]",0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,"[agg., s.m.]",11,THIRD
4,{'$oid': '648aafa169ad0a50d066264b'},paraviaXML/4648.xml,alzare,[v.tr.],0,"[{'id': '1', 'usage': 'FO', 'field': '', 'gram...",[],FO,[v.tr.],9,THIRD


In [16]:
selected_lemmas_df["cluster"].value_counts()

cluster
THIRD     435
SECOND    407
FOURTH    387
FIRST     309
Name: count, dtype: int64

In [17]:
upperbound: int = 150
sampled_df_parts: list[pd.DataFrame] = []
for c in ["FIRST", "SECOND", "THIRD", "FOURTH"]:
    sampled_df_parts.append(
        selected_lemmas_df[selected_lemmas_df["cluster"] == c].sample(
            n=upperbound, random_state=42
        )
    )

sampled_df: pd.DataFrame = pd.concat(sampled_df_parts)
sampled_df["cluster"].value_counts()

cluster
FIRST     150
SECOND    150
THIRD     150
FOURTH    150
Name: count, dtype: int64

In [18]:
sampled_df["lemma"].value_counts()

lemma
smettere      1
ristorante    1
centinaio     1
esame         1
esistere      1
             ..
lingua        1
giocare       1
rete          1
verde         1
treno         1
Name: count, Length: 600, dtype: int64

In [ ]:
sampled_df.to_json(
    "../resources/sensocomune/tdm.sampled.json", orient="records", index=False
)